# REPSOL Model Training (70/15/15)

**MobileNetV3-Large — 01**

Third transfer-learning architecture, chosen for the REPSOL constraints:

- **CPU training is the bottleneck** — EfficientNet-B0 runs ~16 s/batch, ResNet-50 hit memory limits. MobileNetV3-Large (~5.4M params) is explicitly designed for CPU/edge inference: hard-swish activations, squeeze-excitation blocks, and a lean stem make it the fastest of the three per batch.
- **Small dataset (1,382 train samples, 17x imbalance)** — ResNet-50 (25M params) memorised the training set (93% train vs 63% val by epoch 8). A smaller model has less capacity to overfit.
- **Proven on audio** — MobileNetV3 is a standard backbone in spectrogram-based research (keyword spotting, ESC-50, DCASE acoustic scene classification), performing close to EfficientNet at a fraction of the compute.

**Training setup (carries over the EfficientNet-03 fixes):**

| Setting | Value | Why |
|---------|-------|-----|
| LR | 5e-4 | Same as EfficientNet-03; small model tolerates it well |
| Schedule | OneCycleLR (10% warmup → cosine) | Smooth decay, no plateau traps |
| Loss | Weighted CE + label smoothing 0.1 | Balanced weights (NOT squared) + mild ×1.5 boost on classes 1 & 7 |
| Batch size | 8 | CPU memory safe |
| Epochs / patience | 25 / 6 | Enough budget for the cosine schedule |
| Backbone | MobileNetV3-Large IMAGENET1K_V2 | Full fine-tuning (freeze_backbone=False) |

Pipeline: config → verify data → dependencies → train → evaluate → learning curves.

## 0. Config

In [ ]:
from pathlib import Path
import sys
import torch

# ===== Hyperparameters =====
BATCH_SIZE      = 8
EPOCHS          = 25
LEARNING_RATE   = 5e-4
PATIENCE        = 6
LABEL_SMOOTHING = 0.1
MODEL_NAME      = "mobilenetv3"

# ===== Paths =====
PROJECT_ROOT    = Path(r"D:\Work\Internships\INMAR\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms"
OUTPUT_DIR      = PROJECT_ROOT / "Models_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


def next_run_path(model_name, suffix, ext, output_dir):
    prefix = f"{model_name}{suffix}"
    existing = [0]
    for p in output_dir.iterdir():
        if not p.is_file() or p.suffix != ext:
            continue
        stem = p.stem
        if stem.startswith(prefix + "_"):
            tail = stem[len(prefix) + 1:]
            if tail.isdigit():
                existing.append(int(tail))
    return output_dir / f"{prefix}_{max(existing)+1:02d}{ext}"


CHECKPOINT_PATH = next_run_path(MODEL_NAME, "_best", ".pth", OUTPUT_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE         :", DEVICE)
print(f"LR={LEARNING_RATE}  EPOCHS={EPOCHS}  PATIENCE={PATIENCE}  LABEL_SMOOTHING={LABEL_SMOOTHING}")

## 1. Verify Data

In [ ]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.norm.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("Tensor files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .norm.pt files found."
assert counts["val"]   > 0, "No val .norm.pt files found."
assert counts["test"]  > 0, "No test .norm.pt files found."

## 2. Dependencies

In [ ]:
import importlib, subprocess, sys

required = ["torch", "torchvision", "torchaudio", "scikit-learn", "pandas", "tqdm", "numpy"]
name_map = {"scikit-learn": "sklearn"}
for pkg in required:
    try:
        importlib.import_module(name_map.get(pkg, pkg.replace("-", "_")))
        print(f"OK: {pkg}")
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed: {pkg}")

## 3. Training

In [ ]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm

import src.MobileNet.model as model_module
from src.dataloaders import get_dataloaders

model_module = importlib.reload(model_module)
MobileNetV3Spectrogram = model_module.MobileNetV3Spectrogram
compute_class_weights  = model_module.compute_class_weights

# ── DataLoaders ──
train_loader, val_loader, test_loader = get_dataloaders(
    SPECTROGRAM_DIR, batch_size=BATCH_SIZE,
    num_workers=0, pin_memory=False, persistent_workers=False,
)
NUM_CLASSES = len(train_loader.dataset.classes)
CLASS_NAMES = train_loader.dataset.classes
print("Classes:", NUM_CLASSES, "  Train batches:", len(train_loader))

# ── Model ──
model = MobileNetV3Spectrogram(num_classes=NUM_CLASSES, freeze_backbone=False).to(DEVICE)
param_count = sum(p.numel() for p in model.parameters())
print(f"Parameters: {param_count/1e6:.1f}M")

# ── Class weights: sklearn balanced + ×1.5 boost for classes 1 and 7 ──
BOOST_CLASSES = {1: 1.5, 7: 1.5}

train_labels  = [label for (_, label) in train_loader.dataset.samples]
base_weights  = compute_class_weights(train_labels, num_classes=NUM_CLASSES).numpy()
final_weights = base_weights.copy()
for cls_idx, multiplier in BOOST_CLASSES.items():
    final_weights[cls_idx] *= multiplier
class_weights = torch.tensor(final_weights, dtype=torch.float).to(DEVICE)

print("\nClass weights:")
for i, (name, bw, fw) in enumerate(zip(CLASS_NAMES, base_weights, final_weights)):
    boost = f" ×{BOOST_CLASSES[i]}" if i in BOOST_CLASSES else ""
    print(f"  [{i}] {name[:45]:45s}  balanced={bw:.3f}  final={fw:.3f}{boost}")

# ── Loss: weighted CrossEntropy + label smoothing ──
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

# ── Optimiser ──
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# ── Scheduler: OneCycleLR (linear warmup → cosine decay) ──
total_steps = EPOCHS * len(train_loader)
scheduler = OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=total_steps,
    pct_start=0.1,
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=100,
)

# ── Checkpoint + history setup ──
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
HISTORY_PATH = CHECKPOINT_PATH.with_name(f"{CHECKPOINT_PATH.stem}_training_history.csv")
for p in (CHECKPOINT_PATH, HISTORY_PATH):
    if p.exists(): p.unlink()
with open(HISTORY_PATH, "w") as fh:
    fh.write("epoch,train_loss,val_loss,train_acc,val_acc,lr\n")
torch.save(model.state_dict(), CHECKPOINT_PATH)

# ── Training loop ──
best_val_loss = float("inf")
no_improve = 0

for epoch in range(1, EPOCHS + 1):

    # --- train ---
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} Train", ncols=100, unit="batch")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        t_loss    += loss.item()
        t_correct += (out.argmax(1) == y).sum().item()
        t_total   += y.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    train_loss = t_loss / len(train_loader)
    train_acc  = 100.0 * t_correct / t_total

    # --- validate ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} Val  ", ncols=100, unit="batch")
    with torch.no_grad():
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out  = model(x)
            loss = criterion(out, y)
            v_loss    += loss.item()
            v_correct += (out.argmax(1) == y).sum().item()
            v_total   += y.size(0)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    val_loss = v_loss / len(val_loader)
    val_acc  = 100.0 * v_correct / v_total

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:02d}/{EPOCHS} "
        f"| Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} "
        f"| Train Acc: {train_acc:.2f} | Val Acc: {val_acc:.2f} "
        f"| LR: {current_lr:.2e}",
        flush=True,
    )

    with open(HISTORY_PATH, "a") as fh:
        fh.write(f"{epoch},{train_loss:.6f},{val_loss:.6f},{train_acc:.4f},{val_acc:.4f},{current_lr:.6f}\n")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  ✓ Saved improved checkpoint → {CHECKPOINT_PATH.name}", flush=True)
    else:
        no_improve += 1
        print(f"  No improvement {no_improve}/{PATIENCE}", flush=True)
        if no_improve >= PATIENCE:
            print(f"  Early stopping.", flush=True)
            break

print("\nTraining finished.")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## 4. Evaluation

In [ ]:
import importlib
import src.evaluate as eval_module

eval_module = importlib.reload(eval_module)
evaluate_model = eval_module.evaluate_model

# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

val_metrics  = evaluate_model(model, val_loader,  DEVICE)
test_metrics = evaluate_model(model, test_loader, DEVICE)

print("Validation:")
print({k: round(val_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})
print("\nTest:")
print({k: round(test_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})

In [ ]:
print("Test Classification Report:\n")
print(test_metrics["report"])
print("Confusion Matrix:")
print(test_metrics["confusion_matrix"])

## 5. Learning Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(HISTORY_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(df["epoch"], df["train_acc"], label="train")
axes[0].plot(df["epoch"], df["val_acc"],   label="val", linestyle="--")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(df["epoch"], df["train_loss"], label="train")
axes[1].plot(df["epoch"], df["val_loss"],   label="val", linestyle="--")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(df["epoch"], df["lr"])
axes[2].set_title("Learning Rate (OneCycleLR)"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

fig.suptitle("MobileNetV3-01 Learning Curves", fontsize=13)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "evaluation" / "learning_curves_mobilenetv3_01.png",
            dpi=150, bbox_inches="tight")
plt.show()